In [1]:
import torch
from torch.utils.data import DataLoader
from seqeval.metrics import classification_report

from evaluate import evaluate_bilstm_crf
from models.bilstm_crf import BiLSTM_CRF
from utils.bilstm_crf.data_utils import (
    read_conll_2,
    NERDataset,
    collate_fn
)

In [2]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
BATCH_SIZE = 10

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_PATH = "../data/matscholar/test.txt"
MODEL_PATH = "../models/bilstm_crf.pt"

In [3]:
# 读取模型保存文件
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

word2idx = checkpoint["word2idx"]
tag2idx= checkpoint["tag2idx"]
idx2tag = {v: k for k, v in tag2idx.items()}

# 读取测试集
test_sentence, test_tags = read_conll_2(TEST_PATH)

# 构造测试集 DataLoader
test_data = NERDataset(test_sentence, test_tags, word2idx, tag2idx)
test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

# 建立模型
model = BiLSTM_CRF(
    vocab_size=len(word2idx),
    tag_to_ix=tag2idx,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
).to(DEVICE)

# 加载参数
model.load_state_dict(checkpoint["model_state_dict"])

# 测试评估
test_loss, test_p, test_r, test_f1, y_true, y_pred = evaluate_bilstm_crf(
    model, test_loader, idx2tag, DEVICE
)

print("\n========== Test Result ==========")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Precision: {test_p:.4f}")
print(f"Test Recall: {test_r:.4f}")
print(f"Test F1: {test_f1:.4f}")

print("\n========== Detailed Report ==========")
print(classification_report(y_true, y_pred, digits=4))


========== Test Result ==========
Test Loss: 8.0378
Test Precision: 0.7542
Test Recall: 0.7256
Test F1: 0.7396

========== Detailed Report ==========
              precision    recall  f1-score   support

         APL     0.7903    0.5213    0.6282        94
         CMT     0.8398    0.8261    0.8329       184
         DSC     0.7105    0.8060    0.7552       402
         MAT     0.7877    0.6963    0.7392       698
         PRO     0.7179    0.7214    0.7197       621
         SMT     0.7917    0.6847    0.7343       222
         SPL     0.6889    0.7381    0.7126        42

   micro avg     0.7542    0.7256    0.7396      2263
   macro avg     0.7610    0.7134    0.7317      2263
weighted avg     0.7577    0.7256    0.7387      2263

